# Simulación del juego Othello

## Introducción

En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en el juego Othello. Othello es un juego de estrategia para dos jugadores en el que se colocan fichas en un tablero con el objetivo de tener la mayoría al final de la partida.

Reglas:
 - El juego clásico se juega en un tablero de 8x8 casillas (aunque en esta implementación pueden tomar valores entre 4 y 10).
 - Cada ficha tiene una cara blanca y una negra (en esta implementación se representan con 1 y 2).
 - Comienza el jugador que lleva las fichas negras.
 - Los jugadores se turnan para colocar una ficha con su color hacia arriba.
 - Una jugada válida debe encerrar una o más fichas del oponente entre la nueva ficha y otra del mismo color.
 - Las fichas del oponente que quedan encerradas se voltean al color del jugador que hizo la jugada.
 - El juego termina cuando ningún jugador puede hacer una jugada válida.
 - Gana quien tenga más fichas de su color en el tablero al final de la partida.

 En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en Othello.

In [1]:
import sys
import os
import pandas as pd
import random
import numpy as np

# Fijar la semilla
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

# Agregar los directorios al path para poder importar los módulos
sys.path.append(os.path.abspath("../games"))
sys.path.append(os.path.abspath("../methods"))
sys.path.append(os.path.abspath("../selection"))
sys.path.append(os.path.abspath("../graphviz"))

# Importar la clase OthelloGame y las funciones de los métodos
from othello import OthelloGame
from monte_carlo import MCPlay, MCAgent
from monte_carlo_tree_search import MCTS, MCTSPlay, Node, MCTSAgent
from epsilon_greedy import EpsilonGreedy
from softmax import Softmax
from adaptive_softmax import AdaptiveSoftmax
from ucb1 import UCB1
from ucb2 import UCB2
from gradiente_preferencias import GradienteDePreferencias
from MCTS_graphviz import generate_tree_MCTS

## Implementación de la clase `OthelloGame`

En esta sección, se explica cómo ha sido implementada la clase `OthelloGame` que modela el juego. Para ello, a continuación se presenta una breve descripción de cada uno de los métodos que la componen:

 - `__init__(self, rows=8, cols=8, starting_player=1)`: Se utiliza para inicializar el juego. Inicializa el tablero de tamaño `rows` x `cols` con casillas vacías (0). Sus parámetros incluyen:
   - `rows`: número de filas del tablero (de 4 a 10, debe ser par).
   - `cols`: número de columnas del tablero (de 4 a 10, debe ser par).
   - `starting_player`: el jugador que empieza el juego (1 o -1).

 - `comprobar_limites(self, row, col)`: Este método verifica si la posición `(row, col)` está dentro de los límites del tablero.

 - `fichas_a_voltear_en_direccion(self, row, col, d_row, d_col)`: Determina qué fichas se deben voltear en una dirección específica (vertical, horizontal o diagonal) definida por `d_row` y `d_col`. Estas fichas se corresponden con las fichas del oponente que están entre la ficha colocada y otra ficha propia.

 - `valid_actions(self)`: Este método devuelve las acciones válidas que un jugador puede realizar. Recorre el tablero y verifica para cada casilla vacía si colocar una ficha allí es una jugada válida, esto es, si hay al menos una ficha del oponente que puede ser encerrada entre la ficha colocada y otra ficha del mismo jugador. Las direcciones posibles son: arriba, abajo, izquierda, derecha, y las 4 diagonales.

 - `action(self, coordinates)`: Este método simula un turno del juego en la posición dada por las coordenadas `coordinates`, de la forma `(row, col)`. Coloca la ficha en la casilla indicada y voltea las fichas del oponente que hayan sido encerradas. Finalmente, devuelve el nuevo estado del juego con el tablero actualizado y el turno del siguiente jugador. Además, realiza una comprobación para asegurarse de que la acción solicitada sea válida.

 - `terminal(self)`: Verifica si el juego ha terminado, esto es, si ninguno de los dos jugadores tiene jugadas válidas.

 - `winner(self)`: Devuelve quién es el ganador (el que tiene más fichas en el tablero al finalizar la partida). Si el juego ha terminado, devuelve el número del jugador ganador (1 o -1) o un 0 si ha habido empate, pero si no ha terminado devuelve 0.

 - `draw(self)`: Este método dibuja el estado actual del juego, imprimiendo el tablero en formato de matriz, donde cada celda representa una ficha (1, 2, o 0 si está vacía).

## Simulaciones

En esta sección, se realizan simulaciones con distintas estrategias para la toma de decisiones en el juego. Comenzamos creando una instancia del juego `OthelloGame`, y visualizamos su estado inicial. Por defecto, se juega con un tablero con 8 filas y 8 columnas. Sin embargo, estos parámetros se pueden modificar con `rows` y `cols`.

In [2]:
# Crear una instancia del juego Othello con los valores predeterminados
game = OthelloGame()

# Mostrar el estado inicial del juego
game.draw()

[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 1, 0, 0, 0]
[0, 0, 0, 1, 2, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]


A continuación, se presentan los distintos métodos utilizados para realizar las simulaciones.

## Monte-Carlo

En primer lugar, se implementa el método de Monte-Carlo para estimar las mejores jugadas. 

In [9]:
# Crear una instancia del juego Othello
game = OthelloGame(rows=6, cols=6, starting_player=-1)

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 150

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = eval(input("¿En qué casilla quieres colocar tu ficha? (fila, columna): "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue  # Volver a pedir jugada

    else:  # Turno de la IA (Monte-Carlo)
        print("\nTurno de la computadora...")
        game = MCPlay(game, num_simulations)

    game.draw()  # Mostrar el estado después del turno

# Anunciar el ganador
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 1, 2, 0, 0]
[0, 0, 2, 1, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 1, 2, 0, 0]
[0, 0, 2, 2, 0, 0]
[0, 0, 0, 2, 0, 0]
[0, 0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 0, 1, 2, 0, 0]
[0, 0, 2, 1, 0, 0]
[0, 0, 0, 2, 1, 0]
[0, 0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0]
[0, 2, 2, 2, 0, 0]
[0, 0, 2, 1, 0, 0]
[0, 0, 0, 2, 1, 0]
[0, 0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 2, 1, 2, 0, 0]
[0, 0, 2, 1, 0, 0]
[0, 0, 0, 2, 1, 0]
[0, 0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 2, 1, 2, 0, 0]
[0, 0, 2, 2, 2, 0]
[0, 0, 0, 2, 1, 0]
[0, 0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 2, 1, 2, 0, 0]
[0, 0, 1, 2, 2, 0]
[0, 0, 1, 1, 1, 0]
[0, 0, 0, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0]
[0, 2, 1, 2, 

### Monte-Carlo Tree Search

En esta sección, se utiliza el método de Monte-Carlo Tree Search para estimar las mejores jugadas en el juego Othello. Este algoritmo está compuesto por distintas fases, que son:

 1. **Selección:** En esta fase, se parte del nodo raíz y se desciende por el árbol seleccionando nodos hijos sucesivamente según una estrategia. Esto continúa hasta llegar a un nodo hoja, es decir, un nodo que tiene al menos un hijo potencial al que no se le ha aplicado ninguna simulación todavía. Entre las estrategias de selección encontramos: $\epsilon$-greedy, Softmax, Adaptive Softmax, UCB1, UCB2 y Gradiente de Preferencias.

 2. **Expansión:** A partir del nodo hoja seleccionado, se genera un nuevo nodo hijo, es decir, se aplica un movimiento válido que aún no ha sido explorado desde el nodo hoja.

 3. **Simulación:** Desde el nodo hijo recién creado, se completa una jugada aleatoria hasta alcanzar un estado terminal (por ejemplo, una victoria, derrota o empate). Mediante esta simulación, se puede estimar el resultado potencial de seguir esa línea de decisión.

 4. **Retropropagación:** Los resultados de la simulación se propagan hacia atrás, usándose para actualizar las estadísticas de los nodos, que son recorridos desde el nodo expandido hasta el nodo raíz. Esto permite que en futuras decisiones se refuercen las rutas más prometedoras y se descarten las menos efectivas.

In [ ]:
# Crear una instancia del juego Othello
game = OthelloGame()

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 100

# Escoger el método de selección
selection_algorithm_class = GradienteDePreferencias

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = eval(input("¿En qué casilla quieres colocar tu ficha? (fila, columna): "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue

    else:  # Turno de la IA con MCTS
        print("\nTurno de la computadora...")
        game = MCTSPlay(game, num_simulations, selection_algorithm_class)

    game.draw()

# Resultado
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

A continuación, se utiliza `graphviz` como herramienta de depuración, generando un diagrama con el árbol de posiciones siguientes a partir de una determinada posición inicial, dada por un nodo raíz. Cada nodo del árbol representa un estado del juego, el cual se alcanza mediante una secuencia de movimientos, y contiene información como el turno del jugador, el número de visitas y la recompensa de cada jugador. Las aristas indican las acciones tomadas para pasar de un estado al siguiente. Se puede especificar la profundidad máxima del árbol generado.

In [ ]:
# Crear estado inicial del juego
initial_state = OthelloGame()

# Visualizar el árbol de MCTS
root_node = Node(initial_state, None)
mcts = MCTS(root_node, EpsilonGreedy, simulations=100, epsilon=0.2)
mcts.run()
id_to_node = generate_tree_MCTS(root_node, max_depth=2, filename="depuration_trees/arbol_othello", format="pdf")

Los nodos en el árbol aparecen identificados mediante un ID. Para poder visualizar cuál es el estado de cada nodo, la función `generate_tree_MCTS` devuelve un diccionario que asocia cada ID al nodo correspondiente.

In [ ]:
node = id_to_node['nodo1']

node.state.draw()

## Simulaciones comparativas

In [18]:
def play_comparative_game(agent1, agent2, rows=4, cols=4, starting_player=1):
    game = OthelloGame(rows=rows, cols=cols, starting_player=starting_player)
    agents = {1: agent1, -1: agent2}

    while not game.terminal():
        game = agents[game.turn].move(game)

    return game.winner()

In [19]:
def run_simulations(agent_1, agent_2, rows=4, cols=4, n_games=100):
    results = []

    for i in range(n_games):
        # Alternar jugador inicial
        starting_player = 1 if i % 2 == 0 else -1

        # Asignar agentes según quién empieza
        if starting_player == 1:
            winner = play_comparative_game(agent_1, agent_2, rows, cols, starting_player)
        else:
            winner = play_comparative_game(agent_2, agent_1, rows, cols, starting_player)
            # Invertir perspectiva
            winner *= -1

        results.append(winner)

    df = pd.DataFrame(results, columns=["winner"])
    win_rate = (df["winner"] == 1).mean()
    print(f"% de partidas ganadas por el agente 1: {win_rate:.2%} ({df['winner'].value_counts().to_dict()})")
    return df

In [21]:
agent_mc = MCAgent(300)
agent_mcts_eps = MCTSAgent(300, EpsilonGreedy, epsilon = 0.2)
agent_mcts_UCB1 = MCTSAgent(300, UCB1, c=1)
agent_mcts_UCB2 = MCTSAgent(300, UCB2, alpha=0.5)
agent_mcts_soft = MCTSAgent(300, Softmax, tau = 1)
agent_mcts_adapsoft = MCTSAgent(300, AdaptiveSoftmax, tau_0 = 1, alpha = 0.5)
agent_mcts_grad = MCTSAgent(300, GradienteDePreferencias, alpha = 0.2)

### Monte Carlo vs MCTS con $\epsilon$-greedy

In [22]:
df_mc_mcts_eps = run_simulations(agent_mc, agent_mcts_eps, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 1.00% ({-1: 99, 1: 1})


### Monte Carlo vs MCTS con UCB1

In [23]:
df_mc_mcts_UCB1 = run_simulations(agent_mc, agent_mcts_UCB1, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 0.00% ({-1: 100})


### Monte Carlo vs MCTS con UCB2

In [24]:
df_mc_mcts_UCB2 = run_simulations(agent_mc, agent_mcts_UCB2, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 23.00% ({-1: 77, 1: 23})


### Monte Carlo vs MCTS con Softmax

In [25]:
df_mc_mcts_soft = run_simulations(agent_mc, agent_mcts_soft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 0.00% ({-1: 100})


### Monte Carlo vs MCTS con Softmax Adaptativo

In [26]:
df_mc_mcts_adapsoft = run_simulations(agent_mc, agent_mcts_adapsoft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 0.00% ({-1: 100})


### Monte Carlo vs MCTS con Gradiente de Preferencias

In [ ]:
df_mc_mcts_grad = run_simulations(agent_mc, agent_mcts_grad, rows=4, cols=4, n_games=100)

### MCTS con $\epsilon$-greedy vs MCTS con UCB1

In [27]:
df_mcts_eps_UCB1 = run_simulations(agent_mcts_eps, agent_mcts_UCB1, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 0.00% ({-1: 100})


### MCTS con $\epsilon$-greedy vs MCTS con UCB2

In [28]:
df_mcts_eps_UCB2 = run_simulations(agent_mcts_eps, agent_mcts_UCB2, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 46.00% ({-1: 54, 1: 46})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax

In [29]:
df_mcts_eps_soft = run_simulations(agent_mcts_eps, agent_mcts_soft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 24.00% ({-1: 76, 1: 24})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax Adaptativo

In [30]:
df_mcts_eps_adapsoft = run_simulations(agent_mcts_eps, agent_mcts_adapsoft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 28.00% ({-1: 72, 1: 28})


### MCTS con $\epsilon$-greedy vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_eps_grad = run_simulations(agent_mcts_eps, agent_mcts_grad, rows=4, cols=4, n_games=100)

### MCTS con UCB1 vs MCTS con UCB2

In [31]:
df_mcts_UCB1_UCB2 = run_simulations(agent_mcts_UCB1, agent_mcts_UCB2, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 78.00% ({1: 78, -1: 22})


### MCTS con UCB1 vs MCTS con Softmax

In [32]:
df_mcts_UCB1_soft = run_simulations(agent_mcts_UCB1, agent_mcts_soft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 77.00% ({1: 77, -1: 23})


### MCTS con UCB1 vs MCTS con Softmax Adaptativo

In [33]:
df_mcts_UCB1_adapsoft = run_simulations(agent_mcts_UCB1, agent_mcts_adapsoft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 73.00% ({1: 73, -1: 27})


### MCTS con UCB1 vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB1_grad = run_simulations(agent_mcts_UCB1, agent_mcts_grad, rows=4, cols=4, n_games=100)

### MCTS con UCB2 vs MCTS con Softmax

In [34]:
df_mcts_UCB2_soft = run_simulations(agent_mcts_UCB2, agent_mcts_soft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 3.00% ({-1: 97, 1: 3})


### MCTS con UCB2 vs MCTS con Softmax Adaptativo

In [35]:
df_mcts_UCB2_adapsoft = run_simulations(agent_mcts_UCB2, agent_mcts_adapsoft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 3.00% ({-1: 97, 1: 3})


### MCTS con UCB2 vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_UCB2, agent_mcts_grad, rows=4, cols=4, n_games=100)

### MCTS con Softmax vs MCTS con Softmax Adaptativo

In [36]:
df_mcts_soft_adapsoft = run_simulations(agent_mcts_soft, agent_mcts_adapsoft, rows=4, cols=4, n_games=100)

% de partidas ganadas por el agente 1: 15.00% ({-1: 85, 1: 15})


### MCTS con Softmax vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_soft, agent_mcts_grad, rows=4, cols=4, n_games=100)